# 📖 Notebook 2: Cache Coherence & Invalidation

In Notebook 1 we learned how to **partition** data across cache nodes. But what if a node
crashes? What if we want replicas for availability? Now we have **multiple copies** of the
same data, and they can get out of sync.

This notebook covers:
- **Replication**: copying data to multiple nodes for fault tolerance
- **Coherence**: keeping copies in sync when data changes
- **Invalidation**: removing stale data across a multi-node cluster

## Learning Objectives

By the end of this notebook, you'll understand:
- Why replication is needed and the trade-offs between sync/async replication
- Write-invalidate vs write-update strategies
- TTL-based invalidation and its limitations
- How to detect and measure staleness across replicas
- Pub/Sub-based invalidation for real-time consistency

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/distributed-cache
docker-compose up -d
```

### RedisInsight (Redis GUI)
- **URL**: http://localhost:5540
- Add each node: `localhost:6381`, `localhost:6382`, `localhost:6383`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import redis
import time
import threading
import json

# Our 3 cache nodes
NODES = {
    "node-1": redis.Redis(host="localhost", port=6381, decode_responses=True),
    "node-2": redis.Redis(host="localhost", port=6382, decode_responses=True),
    "node-3": redis.Redis(host="localhost", port=6383, decode_responses=True),
}

# Flush all nodes for a clean start
for name, client in NODES.items():
    client.flushall()
    try:
        client.ping()
        print(f"✅ {name} is up")
    except Exception as e:
        print(f"❌ {name} failed: {e}")

## 🔄 Part 1: Why Replicate?

With partitioning, each key lives on **one** node. If that node crashes, the data is gone.

```
Before crash:   node-1: {A, B}    node-2: {C, D}    node-3: {E, F}
Node-2 crashes: node-1: {A, B}    node-2: 💀         node-3: {E, F}
Result:         Keys C and D are LOST → cache misses → database gets hammered
```

**Replication** keeps copies on multiple nodes so data survives failures.

The challenge: how do you keep replicas in sync when data changes?

In [ ]:
# Let's build a simple replicated cache and see the coherence problem

class ReplicatedCache:
    """
    A cache that writes to a primary node and replicates to backup nodes.
    This shows the basic idea of replication — we'll explore different
    replication strategies below.
    """

    def __init__(self, nodes: dict, replication_factor: int = 2):
        self.nodes = nodes
        self.node_names = list(nodes.keys())
        # replication_factor=2 means data lives on 2 nodes (primary + 1 replica)
        self.replication_factor = min(replication_factor, len(nodes))

    def _get_replicas(self, key: str) -> list[str]:
        """Pick which nodes should hold this key (primary + replicas)."""
        import hashlib
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        primary_idx = h % len(self.node_names)
        replicas = []
        for i in range(self.replication_factor):
            idx = (primary_idx + i) % len(self.node_names)
            replicas.append(self.node_names[idx])
        return replicas

    def get(self, key: str) -> str | None:
        """Read from the primary node."""
        replicas = self._get_replicas(key)
        primary = replicas[0]
        return self.nodes[primary].get(key)

    def get_from_any(self, key: str) -> dict:
        """Read from ALL replicas — useful for checking coherence."""
        replicas = self._get_replicas(key)
        results = {}
        for node_name in replicas:
            results[node_name] = self.nodes[node_name].get(key)
        return results


cache = ReplicatedCache(NODES, replication_factor=2)

# Show which nodes each key is assigned to
print("🗂️  Replica assignment (replication factor = 2)")
print("=" * 50)
for key in ["user:alice", "user:bob", "user:charlie"]:
    replicas = cache._get_replicas(key)
    print(f"  {key:<16} → primary: {replicas[0]}, replica: {replicas[1]}")

---

## 🔄 Part 2: Synchronous vs Asynchronous Replication

### Synchronous Replication
Write to **all replicas before** confirming to the client.  
✅ Strong consistency — all replicas always agree  
❌ Slower writes — must wait for the slowest replica  
❌ Lower availability — if any replica is down, writes fail  

### Asynchronous Replication  
Write to the **primary only**, replicate in the background.  
✅ Fast writes — only wait for primary  
✅ Higher availability — writes succeed even if replicas are down  
❌ Eventual consistency — replicas may serve stale data briefly

In [ ]:
# Synchronous Replication: write to ALL replicas before returning

def sync_write(cache: ReplicatedCache, key: str, value: str) -> float:
    """Write to all replicas synchronously. Returns time taken."""
    start = time.time()
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].set(key, value)
    elapsed = (time.time() - start) * 1000
    return elapsed


# Asynchronous Replication: write to primary, replicate later

def async_write(cache: ReplicatedCache, key: str, value: str, delay_ms: int = 100) -> float:
    """Write to primary immediately, replicate to others after a delay."""
    start = time.time()
    replicas = cache._get_replicas(key)

    # Write to primary immediately
    primary = replicas[0]
    cache.nodes[primary].set(key, value)
    elapsed = (time.time() - start) * 1000  # client sees this latency

    # Replicate in background (simulating network delay)
    def replicate_later():
        time.sleep(delay_ms / 1000)
        for node_name in replicas[1:]:
            cache.nodes[node_name].set(key, value)

    threading.Thread(target=replicate_later, daemon=True).start()
    return elapsed


# Compare write latency
print("⏱️  Write Latency Comparison")
print("=" * 50)

sync_times = []
async_times = []

for i in range(100):
    sync_times.append(sync_write(cache, f"bench:sync:{i}", f"value-{i}"))
    async_times.append(async_write(cache, f"bench:async:{i}", f"value-{i}", delay_ms=50))

print(f"  Synchronous:  avg {sum(sync_times)/len(sync_times):.2f} ms per write")
print(f"  Asynchronous: avg {sum(async_times)/len(async_times):.2f} ms per write")
print()
print("  Sync writes are slower because they wait for ALL replicas.")
print("  Async writes return as soon as the primary confirms.")

In [ ]:
# The coherence problem: async replication creates a window of inconsistency

# Flush first
for client in NODES.values():
    client.flushall()

key = "product:price"

# Write with async replication (100ms delay)
async_write(cache, key, "$99.99", delay_ms=500)

# Immediately read from all replicas
print("📖 Reading from all replicas IMMEDIATELY after async write:")
values = cache.get_from_any(key)
for node, val in values.items():
    status = "✅" if val == "$99.99" else "❌ STALE"
    print(f"  {node}: {val!r:>12}  {status}")

print()
print("⏳ Waiting 1 second for replication to complete...")
time.sleep(1)

print()
print("📖 Reading from all replicas AFTER replication:")
values = cache.get_from_any(key)
for node, val in values.items():
    status = "✅" if val == "$99.99" else "❌ STALE"
    print(f"  {node}: {val!r:>12}  {status}")

print()
print("🔑 Key insight: async replication is eventually consistent.")
print("   There's a brief window where replicas disagree.")
print("   For caches, this is usually acceptable — stale data for a few ms")
print("   is better than slow writes or unavailability.")

---

## 🗑️ Part 3: Cache Invalidation Strategies

When the source of truth (database) changes, the cache becomes **stale**.
We need a strategy to remove or update the stale data.

There are three main approaches:

| Strategy | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **TTL-based** | Keys auto-expire after N seconds | Simple, no coordination | Stale for up to TTL |
| **Write-invalidate** | Delete cached key on write | Simple, always fresh on next read | Cache miss after every write |
| **Write-update** | Update cached value on write | Always fresh, no miss | More complex, wasted if rarely read |

In [ ]:
# Strategy 1: TTL-based Invalidation
# Keys automatically expire after a set time

for client in NODES.values():
    client.flushall()

node = NODES["node-1"]

# Set a key with a 3-second TTL
node.set("product:42:price", "$99.99", ex=3)  # ex = expire in N seconds

print("⏰ TTL-based Invalidation Demo")
print("=" * 50)

for i in range(5):
    value = node.get("product:42:price")
    ttl = node.ttl("product:42:price")
    status = "✅ cached" if value else "❌ expired"
    print(f"  t={i}s: value={value!r:<12} TTL={ttl}s  {status}")
    time.sleep(1)

print()
print("💡 After 3 seconds, the key expires automatically.")
print("   Next read will be a cache miss → fetch from DB → re-cache.")
print()
print("   Trade-off: data can be stale for up to TTL seconds.")
print("   Short TTL = fresher data but more DB hits.")
print("   Long TTL = fewer DB hits but staler data.")

In [ ]:
# Strategy 2: Write-Invalidate
# When data changes in the DB, DELETE the cached key on ALL replicas

for client in NODES.values():
    client.flushall()

def write_invalidate(cache: ReplicatedCache, key: str):
    """Delete a key from all replicas that hold it."""
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].delete(key)
    return replicas


# Simulate: write data, then invalidate when DB changes
key = "user:alice:profile"
sync_write(cache, key, json.dumps({"name": "Alice", "email": "alice@old.com"}))

print("1️⃣  Initial state (cached on 2 replicas):")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

# Simulate: Alice changes her email in the database
print("\n2️⃣  Alice changes email in the database...")
print("   DB: {\"name\": \"Alice\", \"email\": \"alice@new.com\"}")

# Invalidate the cached copy
invalidated = write_invalidate(cache, key)
print(f"   Invalidated key on: {invalidated}")

print("\n3️⃣  After invalidation:")
for node, val in cache.get_from_any(key).items():
    status = "(cache miss — will fetch from DB)" if val is None else val
    print(f"   {node}: {status}")

print()
print("✅ Write-invalidate ensures the NEXT read gets fresh data from the DB.")
print("   Downside: that next read is a cache miss (slightly slower).")

In [ ]:
# Strategy 3: Write-Update (push new value to all replicas)

for client in NODES.values():
    client.flushall()

def write_update(cache: ReplicatedCache, key: str, new_value: str):
    """Update a key on all replicas with the new value."""
    replicas = cache._get_replicas(key)
    for node_name in replicas:
        cache.nodes[node_name].set(key, new_value)
    return replicas


key = "user:bob:profile"
sync_write(cache, key, json.dumps({"name": "Bob", "score": 100}))

print("1️⃣  Initial state:")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

# Bob's score changes in the DB
new_data = json.dumps({"name": "Bob", "score": 250})
print(f"\n2️⃣  Bob's score updated in DB to 250...")

# Push the update to all replicas
updated = write_update(cache, key, new_data)
print(f"   Updated cache on: {updated}")

print("\n3️⃣  After write-update:")
for node, val in cache.get_from_any(key).items():
    print(f"   {node}: {val}")

print()
print("✅ Write-update keeps the cache always warm — no cache miss.")
print("   Downside: extra write traffic, wasted if key is rarely read.")

---

## 📡 Part 4: Pub/Sub Invalidation (Cross-Node Coordination)

In a real distributed cache, nodes need to **notify each other** when data changes.
Redis Pub/Sub is a simple way to broadcast invalidation messages.

The pattern:
1. When a key changes, publish an invalidation message to a channel
2. All nodes subscribe to that channel
3. When they receive a message, they delete the local copy

This is how systems like Facebook's Memcached (TAO) coordinate invalidation.

In [ ]:
# Pub/Sub Invalidation Demo
# We use node-1 as the "message bus" for invalidation events

for client in NODES.values():
    client.flushall()

INVALIDATION_CHANNEL = "cache:invalidate"
invalidation_log = []  # track what happened


def start_invalidation_listener(node_name: str, client: redis.Redis):
    """Subscribe to invalidation events and delete local keys."""
    # Use a separate connection for subscribing
    sub_client = redis.Redis(
        host="localhost",
        port=client.connection_pool.connection_kwargs["port"],
        decode_responses=True
    )
    pubsub = sub_client.pubsub()
    pubsub.subscribe(INVALIDATION_CHANNEL)

    def listener():
        for message in pubsub.listen():
            if message["type"] == "message":
                key_to_invalidate = message["data"]
                deleted = client.delete(key_to_invalidate)
                if deleted:
                    invalidation_log.append(f"{node_name}: deleted '{key_to_invalidate}'")

    t = threading.Thread(target=listener, daemon=True)
    t.start()
    return t


# Start listeners on all nodes
listeners = []
for name, client in NODES.items():
    listeners.append(start_invalidation_listener(name, client))

time.sleep(0.5)  # let subscribers connect

# Write the same key to all 3 nodes (simulating full replication)
key = "config:feature_flags"
value = json.dumps({"dark_mode": True, "beta_chat": False})

for client in NODES.values():
    client.set(key, value)

print("1️⃣  Key stored on ALL nodes:")
for name, client in NODES.items():
    print(f"   {name}: {client.get(key)}")

# Now broadcast an invalidation message
print(f"\n2️⃣  Broadcasting invalidation for '{key}'...")
# Publish to the channel on each node so local subscribers hear it
for client in NODES.values():
    client.publish(INVALIDATION_CHANNEL, key)

time.sleep(0.5)  # let the messages propagate

print("\n3️⃣  After invalidation:")
for name, client in NODES.items():
    val = client.get(key)
    status = "✅ invalidated" if val is None else f"⚠️ still has: {val}"
    print(f"   {name}: {status}")

print("\n📋 Invalidation log:")
for entry in invalidation_log:
    print(f"   {entry}")

print()
print("✅ Pub/Sub lets us coordinate invalidation across all nodes in real time.")
print("   In production, you'd use a dedicated message bus (Kafka, Redis Streams, etc.)")

---

## 📏 Part 5: Measuring Staleness

How stale is your cache? Let's build a simple tool to measure the "staleness window" —
the time between a write and when all replicas see the new value.

In [ ]:
# Measure replication lag: how long until all replicas see the update?

for client in NODES.values():
    client.flushall()

def measure_staleness(cache: ReplicatedCache, key: str, value: str,
                      replication_delay_ms: int) -> dict:
    """Write with async replication and measure how long replicas are stale."""
    replicas = cache._get_replicas(key)
    primary = replicas[0]

    # Write to primary
    write_time = time.time()
    cache.nodes[primary].set(key, value)

    # Replicate after delay
    def replicate():
        time.sleep(replication_delay_ms / 1000)
        for node_name in replicas[1:]:
            cache.nodes[node_name].set(key, value)

    threading.Thread(target=replicate, daemon=True).start()

    # Poll replicas until they all have the new value
    stale_durations = {}
    for node_name in replicas[1:]:
        while True:
            if cache.nodes[node_name].get(key) == value:
                stale_durations[node_name] = (time.time() - write_time) * 1000
                break
            time.sleep(0.01)  # poll every 10ms

    return stale_durations


print("📏 Measuring Staleness Window")
print("=" * 50)

for delay in [50, 100, 200, 500]:
    key = f"stale_test:{delay}"
    durations = measure_staleness(cache, key, f"value-{delay}", delay)
    for node, ms in durations.items():
        print(f"  Delay={delay:>3}ms → {node} was stale for {ms:.0f}ms")

print()
print("🔑 The staleness window = replication delay + network latency.")
print("   In production, this is typically 1-10ms within a data center.")
print("   For caches, this is usually acceptable — we trade consistency for speed.")

---

## 🧪 Try It Yourself

1. **Change the replication factor** from 2 to 3 — now every key lives on all nodes.
   How does this affect write latency for synchronous replication?

2. **Simulate a node failure**: stop one Redis container (`docker stop dcache-redis-2`)
   and try reading keys. Does the cache still work with replication factor 2?

3. **Combine TTL + Write-Invalidate**: Set keys with a 60s TTL, but also invalidate
   on write. This gives you a safety net — even if invalidation fails, the TTL
   ensures data refreshes eventually.

4. **Open RedisInsight** and watch keys appear/disappear as you run invalidation.

## 📝 Key Takeaways

| Concept | Key Insight |
|---------|-------------|
| Replication | Copies data to survive node failures |
| Sync replication | Strong consistency, slower writes |
| Async replication | Fast writes, brief staleness window |
| TTL invalidation | Simple, but stale for up to TTL seconds |
| Write-invalidate | Delete on write, cache miss on next read |
| Write-update | Push new value, no cache miss |
| Pub/Sub | Real-time cross-node coordination |

## ➡️ Next: Notebook 3 — Consistent Hashing for Cache Routing

We've seen partitioning (Notebook 1) and coherence (this notebook).  
Now let's deep-dive into **consistent hashing** — the algorithm that makes
distributed caches work in production.

In [ ]:
# Cleanup
for client in NODES.values():
    client.flushall()
print("🧹 All nodes flushed.")